# Predicción de Abandono de Alumnos — Academia Online
### Proyecto final · Scikit Learn + TensorFlow
**Autor:** Diego Martín Pérez

Predecimos qué alumnos **abandonan** (`abandono = 1`) durante las primeras 4 semanas de un curso online, a partir de su perfil y su comportamiento. Comparamos modelos clásicos de Scikit Learn (Regresión Logística y Random Forest) con una red neuronal de TensorFlow, y traducimos los errores del modelo a **coste de negocio**.

> La nota se gana con las **decisiones razonadas**, no con el código. Cada sección termina con la lectura que defenderé.

**Índice**
1. Exploración inicial (EDA)
2. Visualización: tres gráficos justificados
3. Preprocesado + split estratificado
4. Baseline sklearn: LogReg + RandomForest
5. Matriz de confusión y decisión de negocio
6. Red neuronal con TensorFlow (Sequential)
7. Comparativa final y discusión escrita


## 0. Carga del dataset
Dataset sintético de 1200 alumnos (perfil + comportamiento de las primeras 4 semanas). Las relaciones se generan por **umbrales** (p. ej. el riesgo sube si las horas/semana < 4 o si los días sin conectarse > 7), con nulos realistas en `notas_avg` y `nivel_previo`.

In [ ]:
import pandas as pd
import numpy as np

# Dataset sintético: alumnos de una academia online (primeras 4 semanas)
np.random.seed(42)
n = 1200

edad = np.clip(np.random.normal(28, 7, n), 16, 60).round(0).astype(int)
nivel_previo = np.random.choice(['ninguno', 'basico', 'medio', 'avanzado'], size=n, p=[0.45, 0.30, 0.18, 0.07])
modalidad = np.random.choice(['part_time', 'full_time'], size=n, p=[0.62, 0.38])
horas_semana = np.clip(np.random.normal(8, 4, n), 0, 30).round(1)
ejercicios_completados = np.clip(np.random.poisson(12, n), 0, 60)
notas_avg = np.clip(np.random.normal(6.5, 1.8, n), 0, 10).round(1)
asistencias_directos = np.clip(np.random.poisson(3, n), 0, 12)
preguntas_foro = np.clip(np.random.poisson(2, n), 0, 30)
dias_desde_ultima_conexion = np.clip(np.random.exponential(4, n), 0, 30).round(0).astype(int)

# Probabilidad de abandono modelada con señales realistas (por umbrales)
prob = 0.18
prob = prob + np.where(horas_semana < 4, 0.30, 0.0)
prob = prob + np.where(ejercicios_completados < 5, 0.25, 0.0)
prob = prob + np.where(dias_desde_ultima_conexion > 7, 0.30, 0.0)
prob = prob + np.where(notas_avg < 5, 0.20, 0.0)
prob = prob - np.where(asistencias_directos >= 4, 0.15, 0.0)
prob = prob - np.where(preguntas_foro >= 3, 0.10, 0.0)
prob = prob + np.where(nivel_previo == 'ninguno', 0.05, 0.0)
prob = np.clip(prob, 0.02, 0.95)
abandono = (np.random.random(n) < prob).astype(int)

# Huecos realistas
notas_avg_nan = notas_avg.copy()
notas_avg_nan[np.random.choice(n, size=64, replace=False)] = np.nan
nivel_previo_nan = nivel_previo.copy().astype(object)
nivel_previo_nan[np.random.choice(n, size=18, replace=False)] = np.nan

df = pd.DataFrame({
    'alumno_id': range(1, n+1),
    'edad': edad,
    'nivel_previo': nivel_previo_nan,
    'modalidad': modalidad,
    'horas_semana': horas_semana,
    'ejercicios_completados': ejercicios_completados,
    'notas_avg': notas_avg_nan,
    'asistencias_directos': asistencias_directos,
    'preguntas_foro': preguntas_foro,
    'dias_desde_ultima_conexion': dias_desde_ultima_conexion,
    'abandono': abandono,
})

print(f"Dataset cargado: {df.shape[0]} alumnos, {df.shape[1]} columnas")
print(f"Tasa de abandono: {df['abandono'].mean():.1%}")
print(f"\nValores nulos:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

## 1. Exploración inicial (EDA)
Auditamos los datos antes de modelar: estructura, tipos, nulos, tasa de abandono global y por grupos, y correlaciones. **La exploración no es opcional**: aquí se detectan los nulos, el desequilibrio de clases y las señales fuertes que justifican el resto del proyecto.

In [ ]:
print('=== Estructura ===')
print(df.head())
print('\n=== Tipos ===')
print(df.dtypes)
print('\n=== Nulos por columna ===')
print(df.isnull().sum())
print(f'\nAbandono global: {df["abandono"].mean():.1%}')
print('\n=== Abandono por nivel_previo ===')
print(df.groupby('nivel_previo')['abandono'].mean().round(3))
print('\n=== Abandono por modalidad ===')
print(df.groupby('modalidad')['abandono'].mean().round(3))
print('\n=== Correlaciones ===')
print(df[['horas_semana','ejercicios_completados','notas_avg','dias_desde_ultima_conexion','abandono']].corr().round(2))

### Lectura del EDA (para defender)
- **Desequilibrio de clases:** solo el **24.8%** abandona. Es la clase minoritaria → el *accuracy* será engañoso y habrá que mirar recall/F1.
- **Nulos:** `notas_avg` (64) y `nivel_previo` (18) → hay que **imputar** (numérica con mediana, categórica con la moda).
- **Señal más fuerte:** `dias_desde_ultima_conexion` (corr **+0.20** con abandono). La inactividad es el mejor predictor temprano.
- **`nivel_previo`** discrimina: *avanzado* abandona mucho menos (**16.5%**) que el resto (23–29%).
- **`modalidad` NO aporta señal:** part_time 24.7% vs full_time 24.8% → prácticamente idénticas. Variable candidata a descartar.
- **Ojo con las correlaciones lineales bajas** (`horas` −0.12, `notas` −0.11): las relaciones reales son **por umbrales** (el riesgo se dispara solo por debajo de cierto valor), y Pearson no capta bien esa no linealidad. Esto anticipa que un modelo de árboles *podría* ayudar.


## 2. Visualización: tres gráficos justificados
Cada gráfico responde a una pregunta concreta de negocio que defenderé después:
1. **¿Estudiar pocas horas se asocia a abandonar?** → histograma de `horas_semana` por clase.
2. **¿El nivel previo se relaciona con el rendimiento?** → boxplot de `notas_avg` por `nivel_previo`.
3. **¿La inactividad predice el abandono?** → tasa de abandono por tramos de días sin conectarse.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1) Histograma horas_semana por abandono
for t, color in [(0, '#3b82f6'), (1, '#ef4444')]:
    axes[0].hist(df[df['abandono']==t]['horas_semana'], bins=20, alpha=0.6,
                 label=f'abandono={t}', color=color)
axes[0].set_xlabel('horas/semana'); axes[0].set_ylabel('alumnos')
axes[0].set_title('Horas semanales por abandono'); axes[0].legend()

# 2) Boxplot notas_avg por nivel_previo
df.boxplot(column='notas_avg', by='nivel_previo', ax=axes[1])
axes[1].set_title('Notas por nivel previo'); axes[1].set_xlabel('nivel previo'); axes[1].set_ylabel('nota media')

# 3) Tasa de abandono por tramo de inactividad
buckets = pd.cut(df['dias_desde_ultima_conexion'], bins=[-1,1,4,7,40], labels=['0-1','2-4','5-7','8+'])
tasa = df.groupby(buckets, observed=False)['abandono'].mean()
axes[2].bar(tasa.index.astype(str), tasa.values, color='#8b5cf6')
axes[2].set_xlabel('dias desde ultima conexion'); axes[2].set_ylabel('tasa abandono')
axes[2].set_title('Riesgo segun inactividad')

plt.suptitle('')
plt.tight_layout()
plt.show()
print('Tasa de abandono por tramo de inactividad:')
print(tasa.round(3))

### Lectura de los gráficos (para defender)
1. **Horas/semana:** la masa roja (abandono=1) se concentra en valores bajos. Estudiar poco la primera semana es señal de fuga.
2. **Notas por nivel previo:** *avanzado* tiene la mediana de notas más alta y menos cola baja → coherente con que abandone menos.
3. **Inactividad (el gráfico que más defiende):** la tasa de abandono pasa de **~19% (0–1 días)** a **~47% (8+ días)**. Más de una semana sin conectarse **duplica** el riesgo. Es el principal candidato a *alerta temprana* accionable por la academia.


## 3. Preprocesado + split estratificado
Montamos un `ColumnTransformer` que aplica un pipeline distinto a numéricas y categóricas, y partimos en train/test **de forma estratificada**.

**Por qué Pipeline + ColumnTransformer:** encapsula imputación + escalado + codificación, y al hacer `fit` **solo sobre train** evitamos *data leakage* (que información del test se cuele en el entrenamiento). El split estratificado mantiene el 24.8% de abandono en train y test.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

num_features = ['edad', 'horas_semana', 'ejercicios_completados', 'notas_avg',
                'asistencias_directos', 'preguntas_foro', 'dias_desde_ultima_conexion']
cat_features = ['nivel_previo', 'modalidad']  # alumno_id NO es feature

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

X = df[num_features + cat_features]
y = df['abandono']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'train={len(X_train)}  test={len(X_test)}')
print(f'tasa_train={y_train.mean():.2%}  tasa_test={y_test.mean():.2%}')

## 4. Baseline sklearn: LogReg + RandomForest
Antes de redes neuronales, fijamos un **suelo y un techo** con métodos clásicos: un modelo lineal (Regresión Logística) y uno no lineal (Random Forest). Si RF gana mucho, las interacciones no lineales importan; si empata o pierde, la Logística es preferible por interpretable y rápida.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

modelos = {
    'logreg': LogisticRegression(max_iter=1000, random_state=42),
    'rf': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
}
resultados = {}
for nombre, clf in modelos.items():
    pipe = Pipeline([('pre', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    resultados[nombre] = {
        'acc': accuracy_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'pipe': pipe, 'y_pred': y_pred
    }
    print(f"{nombre}: accuracy={resultados[nombre]['acc']:.3f}  F1={resultados[nombre]['f1']:.3f}")

mejor = max(resultados, key=lambda k: resultados[k]['f1'])
print(f'\nMejor por F1: {mejor}')
print(classification_report(y_test, resultados[mejor]['y_pred'], digits=3))

### Lectura del baseline (para defender)
- **LogReg gana** (F1 ≈ 0.22) a Random Forest (F1 ≈ 0.16). Aunque las señales son por umbrales, hay **pocos positivos** (clase minoritaria) y RF **sobreajusta el ruido** con tan pocos ejemplos de abandono; el modelo lineal y regularizado generaliza algo mejor.
- **Alerta:** ambos tienen accuracy ~0.73–0.76 pero **F1 bajísimo**. El `classification_report` revela un **recall de la clase 1 (abandono) de solo ~0.14**: el modelo acierta a los que se quedan, pero **no detecta a los que abandonan**. Justo el problema que analizamos en la siguiente sección.


## 5. Matriz de confusión y decisión de negocio
El accuracy no captura el **coste real** de cada tipo de error. Desglosamos la matriz del mejor modelo y traducimos los errores a euros.

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred_mejor = resultados[mejor]['y_pred']
cm = confusion_matrix(y_test, y_pred_mejor)
print('Matriz de confusión:')
print(cm)

tn, fp, fn, tp = cm.ravel()
print(f'\nTN={tn}  FP={fp}  FN={fn}  TP={tp}')

precision = tp/(tp+fp) if (tp+fp) else 0
recall    = tp/(tp+fn) if (tp+fn) else 0
f1        = 2*precision*recall/(precision+recall) if (precision+recall) else 0
print(f'precision={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}')

### Decisión de negocio (para defender)
Con el umbral por defecto (0.5), el mejor modelo da algo como **TN=174, FP=7, FN=51, TP=8**:

- **Falso Negativo (FN = 51):** alumnos que **iban a abandonar y el modelo no detecta**. Coste = **matrícula perdida** (cientos de €) + no poder intervenir. **Es el error caro.**
- **Falso Positivo (FP = 7):** alumnos a los que avisamos por error. Coste = **un email automático + unos minutos de tutor**. Es barato.

El modelo solo caza **8 de 59 abandonos (recall 14%)**: inservible para el objetivo de negocio aunque su accuracy sea del 76%. **En este problema priorizamos recall sobre precision**, porque el FN es mucho más caro que el FP. La sección 7 propone cómo arreglarlo.


## 6. Red neuronal con TensorFlow (Sequential)
Una red densa sencilla (API `Sequential`): `Dense(32, relu) → Dense(16, relu) → Dense(1, sigmoid)`. La última capa con **sigmoide** + `binary_crossentropy` da la probabilidad de abandono. El objetivo es traducir el problema a una red, no batir al baseline a toda costa.

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

tf.random.set_seed(42)
np.random.seed(42)

# Transformamos a arrays densos (fit en train, transform en test -> sin leakage)
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)
if hasattr(X_train_proc, 'toarray'):
    X_train_proc = X_train_proc.toarray()
    X_test_proc  = X_test_proc.toarray()

model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train_proc.shape[1],)),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(X_train_proc, y_train.values,
                    validation_split=0.2, epochs=20, batch_size=32, verbose=0)

loss, acc = model.evaluate(X_test_proc, y_test.values, verbose=0)
y_pred_nn = (model.predict(X_test_proc, verbose=0) > 0.5).astype(int).ravel()
from sklearn.metrics import f1_score
f1_nn = f1_score(y_test, y_pred_nn)
print(f'NN test accuracy={acc:.3f}  F1={f1_nn:.3f}')

In [ ]:
# Curva de pérdida (diagnóstico de overfitting)
plt.figure(figsize=(8,4))
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.xlabel('época'); plt.ylabel('loss'); plt.title('Curva de pérdida'); plt.legend()
plt.show()

from sklearn.metrics import confusion_matrix
print('Matriz de confusión NN:')
print(confusion_matrix(y_test, y_pred_nn))

### Lectura de la red (para defender)
- La red llega a **accuracy ≈ 0.75 / F1 ≈ 0.21–0.23**, en **empate técnico** con la Regresión Logística (la diferencia oscila con cada ejecución porque el entrenamiento de la red no es 100% determinista) y con el **mismo problema de recall** (sigue sin detectar abandonos).
- **Conclusión clave:** más capacidad/complejidad **no arregla el desequilibrio de clases**. Con 13 features y ~1200 filas, la red no aporta nada sobre un modelo lineal, y sí añade coste de cómputo, falta de interpretabilidad y dependencia de TensorFlow.
- La curva de loss (train ≈ val) indica que **no hay overfitting grave**; el cuello de botella no es el modelo, es **cómo tratamos el desequilibrio**.


## ★ Mejora (opcional pero decisiva): atacar el desequilibrio
El enunciado pide los baselines por defecto, pero la verdadera decisión de negocio es **arreglar el recall**. Reentrenamos la Regresión Logística con `class_weight='balanced'` (penaliza más el fallo en la clase minoritaria). No cambia los baselines de arriba; es la propuesta de mejora.

In [ ]:
lr_balanced = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])
lr_balanced.fit(X_train, y_train)
y_pred_bal = lr_balanced.predict(X_test)

from sklearn.metrics import recall_score
print(f"LogReg balanced -> accuracy={accuracy_score(y_test, y_pred_bal):.3f}  "
      f"F1={f1_score(y_test, y_pred_bal):.3f}  recall={recall_score(y_test, y_pred_bal):.3f}")
print('Matriz de confusión (balanced):')
print(confusion_matrix(y_test, y_pred_bal))

**Resultado:** el recall de abandono sube de **~14% a ~56%** (caza ~33 de 59 en vez de 8). El accuracy baja a ~0.65 y aparecen más falsos positivos (~58), pero **son baratos** (emails de tutor). Para la academia, este modelo es **mucho más útil** aunque su accuracy sea peor: detecta a 4 de cada 7 alumnos en riesgo a tiempo de intervenir. *Esto* es optimizar para el negocio, no para la métrica.


## 7. Comparativa final y discusión escrita
Tabla con los tres modelos y la recomendación a dirección. **La nota depende sobre todo de esta discusión**, así que las respuestas citan *mis números*.

In [ ]:
comparativa = pd.DataFrame([
    {'modelo': 'LogisticRegression', 'accuracy': resultados['logreg']['acc'], 'F1': resultados['logreg']['f1'],
     'inferencia': 'muy rapida (~us)', 'interpretable': 'alta'},
    {'modelo': 'RandomForest', 'accuracy': resultados['rf']['acc'], 'F1': resultados['rf']['f1'],
     'inferencia': 'media (~ms)', 'interpretable': 'media'},
    {'modelo': 'NN_Sequential', 'accuracy': acc, 'F1': f1_nn,
     'inferencia': 'media-lenta (~ms + overhead)', 'interpretable': 'baja'},
]).sort_values('F1', ascending=False).reset_index(drop=True)

comparativa['accuracy'] = comparativa['accuracy'].round(3)
comparativa['F1'] = comparativa['F1'].round(3)
print(comparativa.to_string(index=False))
# LogReg y la NN quedan en empate tecnico (diferencia de F1 < 0.02 = ruido entre ejecuciones).
# RandomForest queda claramente por debajo. La eleccion se decide por coste/interpretabilidad.
print("\nRecomendacion final: LogisticRegression (con class_weight='balanced'). Ver discusion.")

## Discusión escrita de defensa
*(Respuestas concretas, con mis números.)*

**1. ¿Qué señales encontraste en el EDA y cuál es la más útil?**
La señal más fuerte es la **inactividad**: con 0–1 días sin conectarse la tasa de abandono es ~19.8%, y con 8+ días salta a ~46.8% (corr +0.20). También ayudan `horas_semana` y `nivel_previo` (avanzado abandona 16.5% frente a 23–29% del resto). En cambio, `modalidad` no discrimina (24.7% vs 24.8%) y `ejercicios_completados` apenas correlaciona linealmente, porque su efecto es por umbral (< 5 ejercicios), no lineal.

**2. ¿Por qué split estratificado?**
Porque la clase positiva es minoritaria (24.8%). Sin `stratify=y`, un split aleatorio podría dejar train y test con proporciones distintas de abandono y sesgar la evaluación. Con estratificación, train y test mantienen ~24.8%, así las métricas de test son representativas.

**3. ¿Por qué ColumnTransformer + Pipeline evita el data leakage?**
Porque la imputación (mediana/moda) y el escalado se **ajustan solo con `X_train`** y luego se *aplican* a `X_test`. Si calculara la mediana o la media de escalado sobre todo el dataset, estaría filtrando información del test al entrenamiento y las métricas saldrían optimistas y poco realistas.

**4. ¿LogReg o RandomForest? ¿Por qué gana el lineal?**
Gana LogisticRegression (F1 0.216 vs 0.156). Aunque las relaciones son por umbrales (terreno de árboles), hay **pocos positivos**, y RandomForest sobreajusta ese ruido con 200 árboles sin regularizar. El modelo lineal regularizado generaliza mejor, y además es interpretable y rápido. Si tuviéramos muchos más datos de abandono, RF/boosting podría adelantar.

**5. Matriz de confusión: ¿qué error es más caro?**
El **falso negativo**. Un FN (no detectar a un alumno que abandona) cuesta una **matrícula perdida** y la oportunidad de retenerlo; un FP (avisar a alguien que se iba a quedar) cuesta un email y minutos de tutor. Con el umbral por defecto el modelo tiene recall 0.14 (caza 8 de 59 abandonos, 51 FN): inaceptable para el negocio pese al 76% de accuracy.

**6. ¿La red neuronal mejora el baseline? ¿Merece la pena?**
No de forma significativa. La NN da accuracy ~0.75 y F1 ~0.21–0.23, en **empate técnico** con LogReg: la diferencia es menor que la variabilidad entre ejecuciones de la propia red, así que no es un motivo real para elegirla. Mantiene el mismo recall bajo. La curva de loss (train ≈ val) descarta overfitting grave: el problema no es la capacidad del modelo sino el **desequilibrio**. Añadir TensorFlow aquí solo suma coste, latencia y opacidad sin ganancia → no se justifica.

**7. ¿Qué modelo recomiendas a dirección?**
Una **Regresión Logística con `class_weight='balanced'`**. Sube el recall de abandono de ~14% a ~56% (caza ~33 de 59), a costa de más falsos positivos baratos. Es interpretable (coeficientes explicables a negocio), de inferencia casi instantánea y fácil de mantener. La mejor F1 bruta no es el criterio único: para retención, **maximizar recall a coste de FP baratos** es la decisión correcta, y una diferencia de 0.01–0.02 en F1 entre LogReg y la NN es ruido, no razón para asumir la complejidad de una red. Acompañaría el modelo de una **alerta por inactividad > 7 días** como regla simple y transparente de apoyo.

> **Síntesis:** el accuracy del 76% era un espejismo creado por el desequilibrio. La buena decisión no fue un modelo más potente, sino **alinear la métrica con el coste de negocio** (recall sobre FN caros) y elegir el modelo simple, interpretable y barato que lo consigue.
